Imlo coursework

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [3]:
#defining the train and val
train_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = transform,
    download = True
)

#splitting trainval
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_data, val_data = torch.utils.data.random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) #makes sure that the images used in train and val stick tg
)

#dataloaders for train and val
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=32, shuffle=True, num_workers=2)

100%|██████████| 792M/792M [00:43<00:00, 18.0MB/s] 
100%|██████████| 19.2M/19.2M [00:01<00:00, 17.4MB/s]


In [6]:
image, label = train_data[0]

In [7]:
image.size()

torch.Size([3, 128, 128])

In [9]:
# Adding names for the catergories
class_names = train_data.dataset.classes

In [ ]:
# Defining the layers
class NeuralNet(nn.Module):
    def __init__(self):
    #calls constructor from nn.Module
    super().__init__()

    self.conv1 = nn.Conv2d(3, 12, 5)
    self.pool = nn.MaxPool2d(2, 2)
    self.conv2 = nn.Conv2d(12, 24, 5)

    self.fc1 = nn.Linear(24 * 29 * 29, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 37)

  def forward(self, input):
    input = self.pool(F.relu(self.conv1(input)))  #applying conv1, then RELU, then pooling layer
    input = self.pool(F.relu(self.conv2(input)))  #applying conv2, then RELU then pooling layer
    input = torch.flatten(input, 1)  #flattening
    input = F.relu(self.fc1(input))  #applying fc1, then RELU
    input = F.relu(self.fc2(input))  #applying fc2, then RELU
    input = self.fc3(input)  #applying fc3
    return input

In [ ]:
# defining the NN itself
network = NeuralNet()
loss_func = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(network.parameters(), lr=0.001)